# D/D/2 (max 200 m)

Twenty vessel arrivals, two service slots and a shared 200 m quay. The only Excel input is the **info** worksheet (first arrival, interarrival time, service time, ship length, maximum wainting time); an arbitrary number of visits is generated below.

Customers (vessels) and the service station (quay) are composed of mix-ins, in the same way as objects in OpenTNSim.

In [1]:
# package(s) for data handling
import pandas as pd
from IPython.display import display

# OpenQTSim mixins and utils
import openqtsim
from openqtsim.core import Identifiable, HasResource, HasLength, ExtraMetadata
from openqtsim.core import HasArrivalSchedule, HasHandlingTimes, HasMaxWait
from openqtsim.core import create_object, inspect_object
from openqtsim.visualizations import plot_gantt

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

This notebook is executed with OpenQTSim version v0.5.1


#### 0. Define input

Use **exactly one** stopping limit: `nr_arrivals=20` and `max_duration_days=None`, or `nr_arrivals=None` and e.g. `max_duration_days=60`. A duration limit includes arrivals at the boundary; service and waiting then drain to completion or reneging, even beyond that limit.

`D` uses fixed times. `M` samples independent negative-exponential times. 

Stochastic first arrivals have mean `AT`; subsequent return intervals have mean `IAT`, separately for each vessel. Stochastic service has vessel-specific mean `loading + unloading`. Arrival and service seeds must be different; changing the service seed does not change the generated arrivals.

These are vessel-specific renewal streams and service mixtures, not necessarily a homogeneous textbook M/M/2 queue. Samples are generated in advance and replayed through the simulator's deterministic-table interface to preserve vessel identity.

In [ ]:
example_title = "D D 2 (max 200 m)"

nr_arrivals = 20
max_duration_days = None

arrival_distribution = "D"
service_distribution = "D"
arrival_seed = 42
service_seed = 43

#### 1. Create service station
Next we create a Quay-object (our service station) out of the available mix-ins:

In [3]:
# make your preferred Quay class out of available mix-ins.
Quay = create_object(
    "Quay",
    (
        Identifiable,  # allows to give the object a name and a random ID
        HasResource,   # number of service slots (c in Kendall's notation)
        HasLength,     # quay length that is shared by the vessels at the quay
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [4]:
inspect_object(Quay, show_parameter_table=True)

,parameter,class,default,status
0,length,HasLength,,mandatory
1,name,Identifiable,,mandatory
2,nr_resources,HasResource,1,optional (unused)
3,id,Identifiable,None,optional (unused)


We define the required input and create the quay object. A vessel is only admitted when both a service slot and enough remaining quay length are available:

In [ ]:
data_quay = {
    "name": "Quay",       # required by Identifiable
    "length": 200,        # required by HasLength: 200 m quay length
    "nr_resources": 2,    # optional for HasResource: two service slots
}

quay = Quay(**data_quay)

#### 2. Create vessels

All times in `info` are **days**, and lengths are metres. `AT` is the first arrival time (or its mean when stochastic); `IAT` is each vessel's return interval (or its mean). Service time is calculated from `loading + unloading`, not the redundant `ST` column. The generated simulation tables use **hours**.

In [6]:
info = pd.read_excel('1002 - D D 2 (max 200 m).xlsx', sheet_name='info')
info

,name,IAT,ST,AT,length,maxwait,unloading,loading
0,Yellowstar,7,3,2,80,1,2,1
1,ABC,30,7,12,100,2,4,3
2,Oil Trade,18,2,5,100,1,2,0
3,Gen. Cargo Trade,60,4,24,160,2,3,1
4,Fruit Trade,10,6,9,90,0,3,3
5,United,16,5,2,80,2,1,4


We make a Vessel class out of the available mix-ins:

In [7]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel",
    (
        Identifiable,        # allows to give the object a name and a random ID
        HasLength,           # vessel length (m) that occupies the quay
        HasArrivalSchedule,  # first arrival AT and return interval IAT (days)
        HasHandlingTimes,    # unloading and loading time (days); service time is their sum
        HasMaxWait,          # maximum waiting time (days) before the vessel reneges
        ExtraMetadata,       # stores all other inputs (here the redundant ST column) as metadata
    ),
)

We can inspect the object to determine which inputs are required and optional:

In [8]:
inspect_object(Vessel, show_parameter_table=True)

,parameter,class,default,status
0,AT,HasArrivalSchedule,,mandatory
1,IAT,HasArrivalSchedule,,mandatory
2,length,HasLength,,mandatory
3,maxwait,HasMaxWait,,mandatory
4,name,Identifiable,,mandatory
5,loading,HasHandlingTimes,0,optional (unused)
6,unloading,HasHandlingTimes,0,optional (unused)
7,id,Identifiable,None,optional (unused)


Each row of the info sheet is a data dict for one vessel object. The redundant `ST` column ends up in `vessel.metadata`:

In [9]:
vessels = [Vessel(**data_vessel) for data_vessel in info.to_dict('records')]

pd.DataFrame([{
    "name": vessel.name, "length": vessel.length, "AT": vessel.AT, "IAT": vessel.IAT,
    "service_time": vessel.service_time, "maxwait": vessel.maxwait, "metadata": vessel.metadata,
} for vessel in vessels])

,name,length,AT,IAT,service_time,maxwait,metadata
0,Yellowstar,80,2,7,3,1,{'ST': 3}
1,ABC,100,12,30,7,2,{'ST': 7}
2,Oil Trade,100,5,18,2,1,{'ST': 2}
3,Gen. Cargo Trade,160,24,60,4,2,{'ST': 4}
4,Fruit Trade,90,9,10,6,0,{'ST': 6}
5,United,80,2,16,5,2,{'ST': 5}


#### 3. Generate the arrival and service tables

Each row represents one visit. Visits are ordered by arrival time, with ties resolved by the original info-sheet row order. The simulator pairs both tables by row position. Subsequent visits remain scheduled even if an earlier visit reneges. `maxwait` is converted from days to hours, without randomisation.

In [10]:
arrival_table, service_table = openqtsim.generate_vessel_tables(
    vessels,
    nr_arrivals=nr_arrivals,
    max_duration_days=max_duration_days,
    arrival_distribution=arrival_distribution,
    service_distribution=service_distribution,
    arrival_seed=arrival_seed,
    service_seed=service_seed,
)
print(f'Generated {len(arrival_table)} vessel arrivals; all table times are hours.')
display(arrival_table)
display(service_table)

Generated 20 vessel arrivals; all table times are hours.


,name,vessel_id,visit,IAT,AT,length,maxwait
0,Yellowstar,1,1,48.0,48.0,80,24
1,United,6,1,0.0,48.0,80,48
2,Oil Trade,3,1,72.0,120.0,100,24
3,Yellowstar,1,2,96.0,216.0,80,24
4,Fruit Trade,5,1,0.0,216.0,90,0
5,ABC,2,1,72.0,288.0,100,48
6,Yellowstar,1,3,96.0,384.0,80,24
7,United,6,2,48.0,432.0,80,48
8,Fruit Trade,5,2,24.0,456.0,90,0
9,Yellowstar,1,4,96.0,552.0,80,24


,name,ST
0,Yellowstar,72.0
1,United,120.0
2,Oil Trade,48.0
3,Yellowstar,72.0
4,Fruit Trade,144.0
5,ABC,168.0
6,Yellowstar,72.0
7,United,120.0
8,Fruit Trade,144.0
9,Yellowstar,72.0


#### 4. Run simulation

Admission requires both a free service slot and enough remaining length. Strict FIFO prevents bypassing a blocked vessel. A vessel that cannot start by `AT + maxwait` reneges and leaves the queue permanently. Service starting exactly at the deadline is allowed; zero patience allows immediate service only. The quay object sets the number of servers and the quay length of the queue.

In [11]:
if arrival_table.empty:
    raise ValueError('No arrivals within the selected duration; increase the limit.')
A = openqtsim.ArrivalProcess('D', arrival_table)
S = openqtsim.ServiceProcess('D', service_table)
q = openqtsim.Queue(A, S, station=quay)
sim = openqtsim.Simulation(q)
sim.run(len(arrival_table))
summary = sim.get_stats()

Departures: 18 served, 2 reneged
Waiting time in units of service time: 0.0808
Observation horizon: 0 to 1200.0000 hours

Rho_system: system utilisation: 0.8800
Rho_server: server utilisation: 0.7800
P_0: probability nobody in the system: 0.1200

L_s: average nr of customers in the system: 1.7000
L_q: average nr of customers in the queue: 0.1400
W_s: average time spent in the system: 102.0000
W_q: average time spent in the queue: 8.4000

IAT: average inter arrival time: 52.8000
ST: average service time: 104.0000



In [12]:
sim.plot_system_state()

#### 5. Inspect output

In [13]:
df, df_sys = sim.return_log()
display(df)
display(df_sys.head(10))

,c_id,name,length,maxwait,status,departure,IAT,ST,AT,TSB,TSE,TCSS,TCWQ,ITS,s_id
0,1,Yellowstar,80,24,served,120.0,48.0,72.0,48.0,48.0,120.0,72.0,0.0,48.0,1.0
1,2,United,80,48,served,168.0,0.0,120.0,48.0,48.0,168.0,120.0,0.0,48.0,2.0
2,3,Oil Trade,100,24,served,168.0,72.0,48.0,120.0,120.0,168.0,48.0,0.0,0.0,1.0
3,4,Yellowstar,80,24,served,288.0,96.0,72.0,216.0,216.0,288.0,72.0,0.0,48.0,2.0
4,5,Fruit Trade,90,0,served,360.0,0.0,144.0,216.0,216.0,360.0,144.0,0.0,48.0,1.0
5,6,ABC,100,48,served,456.0,72.0,168.0,288.0,288.0,456.0,168.0,0.0,0.0,2.0
6,7,Yellowstar,80,24,served,456.0,96.0,72.0,384.0,384.0,456.0,72.0,0.0,24.0,1.0
7,8,United,80,48,served,576.0,48.0,120.0,432.0,456.0,576.0,144.0,24.0,0.0,2.0
9,9,Fruit Trade,90,0,served,600.0,24.0,144.0,456.0,456.0,600.0,144.0,0.0,0.0,1.0
8,11,Oil Trade,100,24,reneged,576.0,0.0,NaN,552.0,NaN,NaN,24.0,24.0,NaN,NaN


,t,c_s,c_q
0,0.0,0,0
1,48.0,1,1
2,48.0,2,2
3,48.0,2,1
4,48.0,2,0
5,120.0,3,1
6,120.0,2,1
7,120.0,2,0
8,168.0,1,0
9,168.0,0,0


#### 6. Gantt chart: arrival, waiting, service and reneging

One row per vessel visit, labelled by vessel name. Orange is waiting, blue is actual service, a green cross marks departure after service, and a **red cross** marks reneging. Reneged visits have no service bar. Hover labels show at most two decimals in hours, with days in parentheses. The table below includes status and actual departure time.

In [14]:
fig = plot_gantt(df, title=f"{example_title}: {len(df)} vessel arrivals")
fig.show()
df.sort_values(["AT", "c_id"])[["c_id", "name", "length", "maxwait", "status", "AT", "TSB", "TSE",
                                "departure", "TCWQ", "ST", "TCSS", "s_id"]]

,c_id,name,length,maxwait,status,AT,TSB,TSE,departure,TCWQ,ST,TCSS,s_id
0,1,Yellowstar,80,24,served,48.0,48.0,120.0,120.0,0.0,72.0,72.0,1.0
1,2,United,80,48,served,48.0,48.0,168.0,168.0,0.0,120.0,120.0,2.0
2,3,Oil Trade,100,24,served,120.0,120.0,168.0,168.0,0.0,48.0,48.0,1.0
3,4,Yellowstar,80,24,served,216.0,216.0,288.0,288.0,0.0,72.0,72.0,2.0
4,5,Fruit Trade,90,0,served,216.0,216.0,360.0,360.0,0.0,144.0,144.0,1.0
5,6,ABC,100,48,served,288.0,288.0,456.0,456.0,0.0,168.0,168.0,2.0
6,7,Yellowstar,80,24,served,384.0,384.0,456.0,456.0,0.0,72.0,72.0,1.0
7,8,United,80,48,served,432.0,456.0,576.0,576.0,24.0,120.0,144.0,2.0
9,9,Fruit Trade,90,0,served,456.0,456.0,600.0,600.0,0.0,144.0,144.0,1.0
11,10,Yellowstar,80,24,served,552.0,576.0,648.0,648.0,24.0,72.0,96.0,2.0
